<a href="https://colab.research.google.com/github/isakibul15/PrivacyClassifier/blob/sakibul/Model_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.pipeline import Pipeline
# from transformers import BertTokenizer, BertForSequenceClassification
from tqdm import tqdm
import torch
from torch.utils.data import DataLoader, TensorDataset
from torch.nn import functional as F
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, accuracy_score
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [4]:
df1 = pd.read_csv('/content/final_data_unbalanced_121374.csv')
df2 = pd.read_csv('/content/test_1594.csv')
df3 = pd.read_csv('/content/val_1595.csv')

In [5]:
pip install gensim

In [6]:
X_train = df1['Processed']
y_train = df1['Label']

X_test = df2['Processed']
y_test = df2['Label']

X_val = df3['Processed']
y_val = df3['Label']

# Converting the text data to strings (if not already in string format)
X_train_str = [' '.join(map(str, words)) if isinstance(words, list) else str(words) for words in X_train]
X_val_str = [' '.join(map(str, words)) if isinstance(words, list) else str(words) for words in X_val]
X_test_str = [' '.join(map(str, words)) if isinstance(words, list) else str(words) for words in X_test]

In [7]:
tfidf_vectorizer = TfidfVectorizer(max_features=5000)

# Fit the vectorizer on training data and transform all datasets
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_str)
X_val_tfidf = tfidf_vectorizer.transform(X_val_str)
X_test_tfidf = tfidf_vectorizer.transform(X_test_str)

# **RNN**

In [8]:
# Load the dataset (replace with the correct dataset file path if needed)
df = pd.read_csv('/content/final_data_unbalanced_121374.csv')

# Ensure that all values in the 'Processed' column are strings
df['Processed'] = df['Processed'].fillna('').astype(str)

# Extract features and labels
X = df['Processed']
y = df['Label']

# Split into train and test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Further split the train set into train and validation (80% train, 20% validation of the train set)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Tokenizing and preparing text data
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)

# Tokenize and pad sequences
X_train_tokenized = tokenizer.texts_to_sequences(X_train)
X_val_tokenized = tokenizer.texts_to_sequences(X_val)
X_test_tokenized = tokenizer.texts_to_sequences(X_test)

max_sequence_length = 10  # Set a maximum length for sequences
X_train_padded = pad_sequences(X_train_tokenized, maxlen=max_sequence_length, padding='post')
X_val_padded = pad_sequences(X_val_tokenized, maxlen=max_sequence_length, padding='post')
X_test_padded = pad_sequences(X_test_tokenized, maxlen=max_sequence_length, padding='post')

print("Data preparation completed successfully.")


Data preparation completed successfully.


In [9]:
# Prepare tokenized sentences for Word2Vec
tokenized_sentences = [sentence.split() for sentence in X_train]

# Train a Word2Vec model
embedding_dim = 300
word2vec_model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=embedding_dim,
    window=5,
    min_count=1,
    workers=4
)

# Build the embedding matrix
word_index = tokenizer.word_index
num_words = len(word_index) + 1
embedding_matrix = np.zeros((num_words, embedding_dim))
for word, i in word_index.items():
    if word in word2vec_model.wv:
        embedding_matrix[i] = word2vec_model.wv[word]

# **RNN Model**

In [10]:

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

# Convert labels to categorical (one-hot encoding)
num_classes = len(label_encoder.classes_)
y_train_categorical = to_categorical(y_train_encoded, num_classes=num_classes)
y_val_categorical = to_categorical(y_val_encoded, num_classes=num_classes)
y_test_categorical = to_categorical(y_test_encoded, num_classes=num_classes)


# Early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Define RNN model
model_rnn = Sequential([
    Embedding(input_dim=num_words,
              output_dim=embedding_dim,
              weights=[embedding_matrix],
              input_length=max_sequence_length,
              trainable=False),
    SimpleRNN(units=256, return_sequences=False),  # Simple RNN Layer
    Dense(256, activation='relu'),  # Fully connected layer
    Dense(num_classes, activation='softmax')  # Output layer
])

# Compile the RNN model
model_rnn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Display model summary
model_rnn.summary()

# Train the RNN model
history_rnn = model_rnn.fit(X_train_padded, y_train_categorical, epochs=50, batch_size=64,
                            validation_data=(X_val_padded, y_val_categorical),
                            verbose=1, callbacks=[early_stopping])

# Evaluate the RNN model on validation set
val_loss_rnn, val_accuracy_rnn = model_rnn.evaluate(X_val_padded, y_val_categorical, verbose=0)
print(f"RNN Validation Accuracy: {val_accuracy_rnn:.4f}")

# Evaluate the RNN model on test set
test_loss_rnn, test_accuracy_rnn = model_rnn.evaluate(X_test_padded, y_test_categorical, verbose=0)
print(f"RNN Test Accuracy: {test_accuracy_rnn:.4f}")


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │       4,700,700 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn (SimpleRNN)               │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4,700,700 (17.93 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 4,700,700 (17.93 MB)

Epoch 1/50
1214/1214 ━━━━━━━━━━━━━━━━━━━━ 45s 35ms/step - accuracy: 0.7388 - loss: 0.6392 - val_accuracy: 0.7943 - val_loss: 0.5156
Epoch 2/50
1214/1214 ━━━━━━━━━━━━━━━━━━━━ 68s 23ms/step - accuracy: 0.8019 - loss: 0.5009 - val_accuracy: 0.8053 - val_loss: 0.4893
Epoch 3/50
1214/1214 ━━━━━━━━━━━━━━━━━━━━ 42s 24ms/step - accuracy: 0.8291 - loss: 0.4365 - val_accuracy: 0.8180 - val_loss: 0.4616
Epoch 4/50
1214/1214 ━━━━━━━━━━━━━━━━━━━━ 45s 27ms/step - accuracy: 0.8555 - loss: 0.3738 - val_accuracy: 0.8368 - val_loss: 0.4212
Epoch 5/50
1214/1214 ━━━━━━━━━━━━━━━━━━━━ 36s 23ms/step - accuracy: 0.8795 - loss: 0.3214 - val_accuracy: 0.8445 - val_loss: 0.4169
Epoch 6/50
1214/1214 ━━━━━━━━━━━━━━━━━━━━ 41s 24ms/step - accuracy: 0.8935 - loss: 0.2833 - val_accuracy: 0.8480 - val_loss: 0.4193
Epoch 7/50
1214/1214 ━━━━━━━━━━━━━━━━━━━━ 41s 23ms/step - accuracy: 0.9080 - loss: 0.2448 - val_accuracy: 0.8427 - val_loss: 0.4364
Epoch 8/50
1214/1214 ━━━━━━━━━━━━━━━━━━━━ 43s 25ms/step - accuracy: 0.9195 -

In [11]:
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, auc

# Get predictions for the test set
y_test_pred = model_rnn.predict(X_test_padded)
y_test_pred_classes = np.argmax(y_test_pred, axis=1)  # Convert probabilities to class predictions

# Decode the predicted and true labels
y_test_true_classes = np.argmax(y_test_categorical, axis=1)  # True labels

# Classification Report
class_report = classification_report(y_test_true_classes, y_test_pred_classes, target_names=label_encoder.classes_, output_dict=True)

# Macro-averaged metrics
precision_macro = class_report["macro avg"]["precision"]
recall_macro = class_report["macro avg"]["recall"]
f1_macro = class_report["macro avg"]["f1-score"]

# ROC-AUC Score (Macro-Average)
roc_auc_macro = roc_auc_score(y_test_categorical, y_test_pred, average="macro", multi_class="ovr")

# Accuracy
accuracy = class_report["accuracy"]

# Prepare Results Table
results_table = {
    "Model": "Recurrent Neural Network (RNN)",
    "Data Samples": len(X_train) + len(X_test) + len(X_val),
    "Class": {cls: {
        "Precision": class_report[cls]["precision"],
        "Recall": class_report[cls]["recall"],
        "F1-Score": class_report[cls]["f1-score"]
    } for cls in label_encoder.classes_},
    "Precision (Macro-Avg)": precision_macro,
    "Recall (Macro-Avg)": recall_macro,
    "F1-Score (Macro-Avg)": f1_macro,
    "ROC-AUC (Macro-Avg)": roc_auc_macro,
    "Accuracy": accuracy,
    "Best Parameters": "N/A (Default parameters used)"
}

# Display results
print(results_table)


759/759 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step
{'Model': 'Recurrent Neural Network (RNN)', 'Data Samples': 121374, 'Class': {0: {'Precision': 0.8843149672788685, 'Recall': 0.7516597882648484, 'F1-Score': 0.8126091173617846}, 1: {'Precision': 0.7741638539429273, 'Recall': 0.7840273461777502, 'F1-Score': 0.779064381658175}, 2: {'Precision': 0.8584485407066053, 'Recall': 0.9112180009783141, 'F1-Score': 0.8840465079490627}}, 'Precision (Macro-Avg)': 0.838975787309467, 'Recall (Macro-Avg)': 0.8156350451403043, 'F1-Score (Macro-Avg)': 0.8252400023230075, 'ROC-AUC (Macro-Avg)': 0.9488980173963723, 'Accuracy': 0.84086508753862, 'Best Parameters': 'N/A (Default parameters used)'}


In [ ]:
# # Tokenizer for text processing
# max_sequence_length = 100  # Set according to your data
# num_words = 10000  # Set the vocabulary size (maximum number of words)
# embedding_dim = 300  # Word2Vec embedding dimension

# tokenizer = Tokenizer(num_words=num_words)
# tokenizer.fit_on_texts(X_train)

# # Convert text to sequences
# X_train_seq = tokenizer.texts_to_sequences(X_train)
# X_val_seq = tokenizer.texts_to_sequences(X_val)
# X_test_seq = tokenizer.texts_to_sequences(X_test)

# # Padding sequences to ensure uniform length
# X_train_padded = pad_sequences(X_train_seq, maxlen=max_sequence_length)
# X_val_padded = pad_sequences(X_val_seq, maxlen=max_sequence_length)
# X_test_padded = pad_sequences(X_test_seq, maxlen=max_sequence_length)

# # Prepare tokenized sentences for Word2Vec
# tokenized_sentences = [sentence.split() for sentence in X_train]

# # Train a Word2Vec model
# word2vec_model = Word2Vec(
#     sentences=tokenized_sentences,
#     vector_size=embedding_dim,
#     window=5,
#     min_count=1,
#     workers=4
# )

# # Build the embedding matrix
# word_index = tokenizer.word_index
# num_words = len(word_index) + 1  # Adding 1 because index starts from 1
# embedding_matrix = np.zeros((num_words, embedding_dim))

# for word, i in word_index.items():
#     if word in word2vec_model.wv:
#         embedding_matrix[i] = word2vec_model.wv[word]

# # Convert labels to categorical (one-hot encoding)
# label_encoder = LabelEncoder()
# y_train_encoded = label_encoder.fit_transform(y_train)
# y_val_encoded = label_encoder.transform(y_val)
# y_test_encoded = label_encoder.transform(y_test)

# num_classes = len(label_encoder.classes_)
# y_train_categorical = to_categorical(y_train_encoded, num_classes=num_classes)
# y_val_categorical = to_categorical(y_val_encoded, num_classes=num_classes)
# y_test_categorical = to_categorical(y_test_encoded, num_classes=num_classes)

# # Early stopping callback
# early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# # Define LSTM model
# model_lstm = Sequential([
#     Embedding(input_dim=num_words,
#               output_dim=embedding_dim,
#               weights=[embedding_matrix],
#               input_length=max_sequence_length,
#               trainable=False),
#     LSTM(units=512, return_sequences=False),
#     Dense(512, activation='relu'),
#     Dense(num_classes, activation='softmax')
# ])

# # Compile and train the LSTM model
# model_lstm.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
# model_lstm.summary()

# history = model_lstm.fit(X_train_padded, y_train_categorical, epochs=50, batch_size=64,
#                          validation_data=(X_val_padded, y_val_categorical),
#                          verbose=1, callbacks=[early_stopping])

# # Evaluate the LSTM model on validation set
# val_loss, val_accuracy = model_lstm.evaluate(X_val_padded, y_val_categorical, verbose=0)
# print(f"LSTM Validation Accuracy: {val_accuracy:.4f}")

# # Evaluate the LSTM model on test set
# test_loss, test_accuracy = model_lstm.evaluate(X_test_padded, y_test_categorical, verbose=0)
# print(f"LSTM Test Accuracy: {test_accuracy:.4f}")


# **LSTM**

In [ ]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.4/383.4 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.6/233.6 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 9.3 MB/s eta 0:00:00


In [ ]:
import numpy as np
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score, accuracy_score
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder

# --- Configuration ---
MAX_SEQUENCE_LENGTH = 100  # Max length of sequences
VOCAB_SIZE = 10000  # Max vocabulary size
EMBEDDING_DIM = 300  # Embedding dimension
LSTM_UNITS = 256  # Number of LSTM units
BATCH_SIZE = 256 # Batch size for training
EPOCHS = 20  # Maximum number of epochs
EARLY_STOPPING_PATIENCE = 3  # Patience for early stopping


# --- Data Preparation ---
def prepare_data(X_train, X_val, X_test, y_train, y_val, y_test):
    # Tokenizer for text processing
    tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
    tokenizer.fit_on_texts(X_train)

    # Convert text to sequences and pad
    X_train_padded = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_SEQUENCE_LENGTH)
    X_val_padded = pad_sequences(tokenizer.texts_to_sequences(X_val), maxlen=MAX_SEQUENCE_LENGTH)
    X_test_padded = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=MAX_SEQUENCE_LENGTH)

    # Encode labels
    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(y_train)
    y_val_encoded = label_encoder.transform(y_val)
    y_test_encoded = label_encoder.transform(y_test)

    num_classes = len(label_encoder.classes_)
    y_train_categorical = to_categorical(y_train_encoded, num_classes=num_classes)
    y_val_categorical = to_categorical(y_val_encoded, num_classes=num_classes)
    y_test_categorical = to_categorical(y_test_encoded, num_classes=num_classes)

    return X_train_padded, X_val_padded, X_test_padded, y_train_categorical, y_val_categorical, y_test_categorical, num_classes, tokenizer.word_index, label_encoder
# --- Model Definition ---
def build_simple_lstm_model(vocab_size, embedding_dim, num_classes):
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=MAX_SEQUENCE_LENGTH),
        LSTM(units=LSTM_UNITS),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

#

In [ ]:


def evaluate_model(model, X_test_padded, y_test_categorical, label_encoder):
    y_test_pred_probs = model.predict(X_test_padded, verbose=0)
    y_test_pred_classes = np.argmax(y_test_pred_probs, axis=1)
    y_test_true_classes = np.argmax(y_test_categorical, axis=1)

    overall_accuracy = accuracy_score(y_test_true_classes, y_test_pred_classes)
    macro_roc_auc = roc_auc_score(y_test_categorical, y_test_pred_probs, average='macro', multi_class='ovr')

    print("\n--- Model Evaluation ---\n")
    for class_index, class_name in enumerate(label_encoder.classes_):
        y_true_binary = (y_test_true_classes == class_index).astype(int)
        y_pred_binary = (y_test_pred_classes == class_index).astype(int)

        precision = precision_score(y_true_binary, y_pred_binary, zero_division=0)
        recall = recall_score(y_true_binary, y_pred_binary, zero_division=0)
        f1 = f1_score(y_true_binary, y_pred_binary, zero_division=0)
        roc_auc = roc_auc_score(y_true_binary, y_test_pred_probs[:, class_index])

        print(f"Class: {class_name}")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall: {recall:.4f}")
        print(f"  F1-Score: {f1:.4f}")
        print(f"  ROC-AUC: {roc_auc:.4f}\n")

    print("Overall Performance:")
    print(f"  Accuracy: {overall_accuracy:.4f}")
    print(f"  Macro-Averaged ROC-AUC: {macro_roc_auc:.4f}")

# --- Main Execution ---

# Prepare data
(
    X_train_padded, X_val_padded, X_test_padded,
    y_train_categorical, y_val_categorical, y_test_categorical,
    num_classes, word_index, label_encoder
) = prepare_data(X_train, X_val, X_test, y_train, y_val, y_test)

# Build and train model
model = build_simple_lstm_model(vocab_size=min(len(word_index) + 1, VOCAB_SIZE), embedding_dim=EMBEDDING_DIM, num_classes=num_classes)
model.summary()

early_stopping = EarlyStopping(monitor='val_loss', patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True)

history = model.fit(X_train_padded, y_train_categorical, epochs=EPOCHS, batch_size=BATCH_SIZE,
                    validation_data=(X_val_padded, y_val_categorical), callbacks=[early_stopping], verbose=1)

# Evaluate the model
evaluate_model(model, X_test_padded, y_test_categorical, label_encoder)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 22s 35ms/step - accuracy: 0.7965 - loss: 0.4946 - val_accuracy: 0.8658 - val_loss: 0.4420
Epoch 2/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 16s 34ms/step - accuracy: 0.9273 - loss: 0.1978 - val_accuracy: 0.8489 - val_loss: 0.4702
Epoch 3/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 21s 35ms/step - accuracy: 0.9503 - loss: 0.1370 - val_accuracy: 0.8483 - val_loss: 0.5663
Epoch 4/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 21s 36ms/step - accuracy: 0.9639 - loss: 0.1018 - val_accuracy: 0.8495 - val_loss: 0.6654

--- Model Evaluation ---

Class: 0
  Precision: 0.8609
  Recall: 0.8106
  F1-Score: 0.8350
  ROC-AUC: 0.9496

Class: 1
  Precision: 0.8037
  Recall: 0.8286
  F1-Score: 0.8159
  ROC-AUC: 0.9427

Class: 2
  Precision: 0.8870
  Recall: 0.8957
  F1-Score: 0.8913
  ROC-AUC: 0.9541

Overall Performance:
  Accuracy: 0.8588
  Macro-Averaged ROC-AUC: 0.9488


# **BILSTM**

In [ ]:
pip install optuna

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, accuracy_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from gensim.models import Word2Vec

# --- Configuration ---
MAX_SEQUENCE_LENGTH = 100  # Max length of input sequences
VOCAB_SIZE = 10000  # Max vocabulary size
EMBEDDING_DIM = 300  # Embedding dimensions (e.g., for GloVe or Word2Vec)
BATCH_SIZE =  256  # Batch size
LSTM_UNITS = 256  # Number of BiLSTM units
EPOCHS = 20  # Number of epochs
EARLY_STOPPING_PATIENCE = 3  # Early stopping patience


# --- Data Preparation ---
def prepare_data(X_train, X_val, X_test, y_train, y_val, y_test):
    # Tokenize and pad sequences
    tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
    tokenizer.fit_on_texts(X_train)

    X_train_padded = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_SEQUENCE_LENGTH)
    X_val_padded = pad_sequences(tokenizer.texts_to_sequences(X_val), maxlen=MAX_SEQUENCE_LENGTH)
    X_test_padded = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=MAX_SEQUENCE_LENGTH)

    # Train Word2Vec embedding
    tokenized_sentences = [sentence.split() for sentence in X_train]
    word2vec_model = Word2Vec(
        sentences=tokenized_sentences,
        vector_size=EMBEDDING_DIM,
        window=5,
        min_count=1,
        workers=4
    )

    # Build embedding matrix
    word_index = tokenizer.word_index
    actual_vocab_size = len(word_index) + 1  # Include the `<OOV>` token
    vocab_size = min(actual_vocab_size, VOCAB_SIZE)  # Respect max VOCAB_SIZE

    embedding_matrix = np.zeros((vocab_size, EMBEDDING_DIM))
    for word, i in word_index.items():
        if i < vocab_size:  # Ensure the index is within the embedding matrix bounds
            if word in word2vec_model.wv:
                embedding_matrix[i] = word2vec_model.wv[word]

    # Encode labels
    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(y_train)
    y_val_encoded = label_encoder.transform(y_val)
    y_test_encoded = label_encoder.transform(y_test)

    num_classes = len(label_encoder.classes_)
    y_train_categorical = to_categorical(y_train_encoded, num_classes=num_classes)
    y_val_categorical = to_categorical(y_val_encoded, num_classes=num_classes)
    y_test_categorical = to_categorical(y_test_encoded, num_classes=num_classes)

    return (X_train_padded, X_val_padded, X_test_padded,
            y_train_categorical, y_val_categorical, y_test_categorical,
            vocab_size, embedding_matrix, num_classes, label_encoder)

# --- Build BiLSTM Model ---
def build_bilstm_model(vocab_size, embedding_dim, embedding_matrix, input_length, num_classes):
    model = Sequential([
        Embedding(input_dim=vocab_size,
                  output_dim=embedding_dim,
                  weights=[embedding_matrix],
                  input_length=input_length,
                  trainable=False),
        Bidirectional(LSTM(units=LSTM_UNITS, return_sequences=False)),
        Dense(256, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model




In [ ]:
# --- Evaluation Function ---
def evaluate_bilstm_model(model, X_test_padded, y_test_categorical, label_encoder):
    y_test_pred_probs = model.predict(X_test_padded, verbose=0)
    y_test_pred_classes = np.argmax(y_test_pred_probs, axis=1)
    y_test_true_classes = np.argmax(y_test_categorical, axis=1)

    overall_accuracy = accuracy_score(y_test_true_classes, y_test_pred_classes)
    macro_roc_auc = roc_auc_score(y_test_categorical, y_test_pred_probs, average='macro', multi_class='ovr')

    print("\n--- BiLSTM Model Evaluation Metrics ---\n")

    # Per-class metrics
    for class_index, class_name in enumerate(label_encoder.classes_):
        y_true_binary = (y_test_true_classes == class_index).astype(int)
        y_pred_binary = (y_test_pred_classes == class_index).astype(int)

        precision = precision_score(y_true_binary, y_pred_binary, zero_division=0)
        recall = recall_score(y_true_binary, y_pred_binary, zero_division=0)
        f1 = f1_score(y_true_binary, y_pred_binary, zero_division=0)
        roc_auc = roc_auc_score(y_true_binary, y_test_pred_probs[:, class_index])

        print(f"Class: {class_name}")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall: {recall:.4f}")
        print(f"  F1-Score: {f1:.4f}")
        print(f"  ROC-AUC: {roc_auc:.4f}\n")

    print("Overall Performance:")
    print(f"  Accuracy: {overall_accuracy:.4f}")
    print(f"  Macro-Averaged ROC-AUC: {macro_roc_auc:.4f}")

# --- Main Execution ---
# Assuming `X_train`, `y_train`, `X_val`, `y_val`, `X_test`, `y_test` are already prepared:
(X_train_padded, X_val_padded, X_test_padded,
 y_train_categorical, y_val_categorical, y_test_categorical,
 vocab_size, embedding_matrix, num_classes, label_encoder) = prepare_data(X_train, X_val, X_test, y_train, y_val, y_test)

# Build and train BiLSTM model
model_bilstm = build_bilstm_model(
    vocab_size=vocab_size,
    embedding_dim=EMBEDDING_DIM,
    embedding_matrix=embedding_matrix,
    input_length=MAX_SEQUENCE_LENGTH,
    num_classes=num_classes
)
model_bilstm.summary()

early_stopping = EarlyStopping(monitor='val_loss', patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True)

history = model_bilstm.fit(
    X_train_padded, y_train_categorical,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_data=(X_val_padded, y_val_categorical),
    callbacks=[early_stopping], verbose=1
)

# Evaluate BiLSTM model
evaluate_bilstm_model(model_bilstm, X_test_padded, y_test_categorical, label_encoder)

# Validation and test evaluation
val_loss, val_accuracy = model_bilstm.evaluate(X_val_padded, y_val_categorical, verbose=0)
print(f"\nValidation Accuracy: {val_accuracy:.4f}")

test_loss, test_accuracy = model_bilstm.evaluate(X_test_padded, y_test_categorical, verbose=0)
print(f"Test Accuracy: {test_accuracy:.4f}")

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ ?                           │       3,000,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional (Bidirectional)        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,000,000 (11.44 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 3,000,000 (11.44 MB)

Epoch 1/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 34s 66ms/step - accuracy: 0.8270 - loss: 0.4407 - val_accuracy: 0.8646 - val_loss: 0.3821
Epoch 2/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 31s 66ms/step - accuracy: 0.9282 - loss: 0.1928 - val_accuracy: 0.8589 - val_loss: 0.4707
Epoch 3/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 42s 68ms/step - accuracy: 0.9678 - loss: 0.0941 - val_accuracy: 0.8539 - val_loss: 0.5466
Epoch 4/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 41s 68ms/step - accuracy: 0.9816 - loss: 0.0522 - val_accuracy: 0.8608 - val_loss: 0.7002

--- BiLSTM Model Evaluation Metrics ---

Class: 0
  Precision: 0.8430
  Recall: 0.8524
  F1-Score: 0.8476
  ROC-AUC: 0.9677

Class: 1
  Precision: 0.8188
  Recall: 0.8286
  F1-Score: 0.8237
  ROC-AUC: 0.9596

Class: 2
  Precision: 0.9057
  Recall: 0.8957
  F1-Score: 0.9007
  ROC-AUC: 0.9659

Overall Performance:
  Accuracy: 0.8683
  Macro-Averaged ROC-AUC: 0.9644

Validation Accuracy: 0.8646
Test Accuracy: 0.8683


# **BILSTM With CBOW**

In [ ]:
from gensim.models import Word2Vec
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, accuracy_score
import numpy as np

# --- Configuration ---
MAX_SEQUENCE_LENGTH = 100  # Max length of input sequences
VOCAB_SIZE = 10000  # Maximum vocabulary size
EMBEDDING_DIM = 300  # Dimension of Word2Vec embeddings
BATCH_SIZE = 256   # Batch size for training
EPOCHS = 20  # Number of training epochs
LSTM_UNITS = 256  # Number of units in LSTM
EARLY_STOPPING_PATIENCE = 3  # Patience for early stopping

# --- Initialize Tokenizer and Prepare Data ---
# Assuming `X_train`, `X_val`, and `X_test` contain the text data
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')  # Initialize the tokenizer
tokenizer.fit_on_texts(X_train)  # Fit the tokenizer on the training data

# Tokenize and pad sequences
X_train_padded = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_SEQUENCE_LENGTH)
X_val_padded = pad_sequences(tokenizer.texts_to_sequences(X_val), maxlen=MAX_SEQUENCE_LENGTH)
X_test_padded = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=MAX_SEQUENCE_LENGTH)

# Prepare labels (Assuming `y_train`, `y_val`, and `y_test` contain the labels)
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()  # Initialize label encoder
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

num_classes = len(label_encoder.classes_)  # Number of unique classes
y_train_categorical = to_categorical(y_train_encoded, num_classes=num_classes)
y_val_categorical = to_categorical(y_val_encoded, num_classes=num_classes)
y_test_categorical = to_categorical(y_test_encoded, num_classes=num_classes)

# --- Tokenize Sentences for Word2Vec ---
# Tokenize sentences for Word2Vec training
tokenized_sentences = [sentence.split() for sentence in X_train]

# --- Train CBOW Word2Vec model ---
cbow_model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=EMBEDDING_DIM,
    window=5,
    min_count=1,
    workers=4,
    sg=0  # CBOW model
)

# --- Build the embedding matrix ---
word_index = tokenizer.word_index
actual_vocab_size = min(len(word_index) + 1, VOCAB_SIZE)  # Limit to VOCAB_SIZE
embedding_matrix = np.zeros((actual_vocab_size, EMBEDDING_DIM))

for word, i in word_index.items():
    if i < actual_vocab_size:  # Ensure index is within the embedding matrix bounds
        if word in cbow_model.wv:
            embedding_matrix[i] = cbow_model.wv[word]

# --- Early stopping callback ---
early_stopping = EarlyStopping(monitor='val_loss', patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True)

# --- Define BiLSTM model ---
model_bilstm = Sequential([
    Embedding(
        input_dim=actual_vocab_size,
        output_dim=EMBEDDING_DIM,
        weights=[embedding_matrix],
        input_length=MAX_SEQUENCE_LENGTH,
        trainable=False  # Set trainable=True if fine-tuning embeddings is desired
    ),
    Bidirectional(LSTM(units=LSTM_UNITS, return_sequences=False)),
    Dense(256, activation='relu'),
    Dense(num_classes, activation='softmax')
])



/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
# Compile the BiLSTM model
model_bilstm.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Display model summary
model_bilstm.summary()

# --- Train the BiLSTM model ---
history_bilstm = model_bilstm.fit(
    X_train_padded, y_train_categorical,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_data=(X_val_padded, y_val_categorical),
    verbose=1, callbacks=[early_stopping]
)

# --- Evaluate the BiLSTM model ---
# Predict on test set
y_test_pred_probs_bilstm = model_bilstm.predict(X_test_padded, verbose=0)  # Predicted probabilities
y_test_pred_classes_bilstm = np.argmax(y_test_pred_probs_bilstm, axis=1)  # Predicted class labels
y_test_true_classes = np.argmax(y_test_categorical, axis=1)  # True class labels

# Overall Metrics
overall_accuracy_bilstm = accuracy_score(y_test_true_classes, y_test_pred_classes_bilstm)
macro_roc_auc_bilstm = roc_auc_score(y_test_categorical, y_test_pred_probs_bilstm, average='macro', multi_class='ovr')

print("\n--- CBOW + BiLSTM Model Evaluation Metrics ---\n")

# Per-Class Metrics
for class_index, class_name in enumerate(label_encoder.classes_):
    # Binary labels for the current class
    y_true_binary = (y_test_true_classes == class_index).astype(int)
    y_pred_binary = (y_test_pred_classes_bilstm == class_index).astype(int)

    # Calculate Metrics
    precision = precision_score(y_true_binary, y_pred_binary, zero_division=0)
    recall = recall_score(y_true_binary, y_pred_binary, zero_division=0)
    f1 = f1_score(y_true_binary, y_pred_binary, zero_division=0)
    roc_auc = roc_auc_score(y_true_binary, y_test_pred_probs_bilstm[:, class_index])
    class_accuracy = accuracy_score(y_true_binary, y_pred_binary)

    # Print Metrics for the Current Class
    print(f"Class: {class_name}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1-Score: {f1:.4f}")
    print(f"  ROC-AUC: {roc_auc:.4f}")
    print(f"  Accuracy: {class_accuracy:.4f}\n")

# Overall Metrics
print("Overall Performance:")
print(f"  Accuracy: {overall_accuracy_bilstm:.4f}")
print(f"  Macro-Averaged ROC-AUC: {macro_roc_auc_bilstm:.4f}")

# Evaluate the BiLSTM model on validation set
val_loss_bilstm, val_accuracy_bilstm = model_bilstm.evaluate(X_val_padded, y_val_categorical, verbose=0)
print(f"\nBiLSTM Validation Accuracy: {val_accuracy_bilstm:.4f}")

# Evaluate the BiLSTM model on test set
test_loss_bilstm, test_accuracy_bilstm = model_bilstm.evaluate(X_test_padded, y_test_categorical, verbose=0)
print(f"BiLSTM Test Accuracy: {test_accuracy_bilstm:.4f}")


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)              │ ?                           │       3,000,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_1 (Bidirectional)      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,000,000 (11.44 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 3,000,000 (11.44 MB)

Epoch 1/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 39s 68ms/step - accuracy: 0.8280 - loss: 0.4327 - val_accuracy: 0.8671 - val_loss: 0.3969
Epoch 2/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 32s 68ms/step - accuracy: 0.9302 - loss: 0.1879 - val_accuracy: 0.8665 - val_loss: 0.4748
Epoch 3/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 41s 69ms/step - accuracy: 0.9674 - loss: 0.0947 - val_accuracy: 0.8608 - val_loss: 0.5215
Epoch 4/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 32s 68ms/step - accuracy: 0.9799 - loss: 0.0588 - val_accuracy: 0.8621 - val_loss: 0.7094

--- CBOW + BiLSTM Model Evaluation Metrics ---

Class: 0
  Precision: 0.8592
  Recall: 0.8329
  F1-Score: 0.8458
  ROC-AUC: 0.9594
  Accuracy: 0.9316

Class: 1
  Precision: 0.8033
  Recall: 0.8071
  F1-Score: 0.8052
  ROC-AUC: 0.9537
  Accuracy: 0.8971

Class: 2
  Precision: 0.8896
  Recall: 0.8994
  F1-Score: 0.8944
  ROC-AUC: 0.9634
  Accuracy: 0.8915

Overall Performance:
  Accuracy: 0.8601
  Macro-Averaged ROC-AUC: 0.9588

BiLSTM Validation Accuracy: 0.8671
BiLSTM Test A

# **BILSTM With CBOW + Attention**

In [ ]:
from gensim.models import Word2Vec
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Bidirectional, LSTM, Dense, Embedding, GlobalAveragePooling1D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, accuracy_score
import numpy as np

# --- Configuration ---
EMBEDDING_DIM = 300  # Embedding dimensions
LSTM_UNITS = 512  # LSTM units
MAX_SEQUENCE_LENGTH = 100  # Maximum sequence length
VOCAB_SIZE = 10000  # Maximum vocabulary size
BATCH_SIZE = 256  # Batch size
EPOCHS = 20  # Number of epochs
EARLY_STOPPING_PATIENCE = 3  # Early stopping patience

# --- Step 1: Train CBOW Word2Vec model ---
# Assuming `tokenized_sentences` is a list of tokenized sentences from `X_train`
cbow_model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=EMBEDDING_DIM,
    window=5,
    min_count=1,
    workers=4,
    sg=0  # CBOW model
)

# --- Step 2: Build the embedding matrix ---
word_index = tokenizer.word_index
actual_vocab_size = min(len(word_index) + 1, VOCAB_SIZE)  # Limit vocab size to VOCAB_SIZE
embedding_matrix = np.zeros((actual_vocab_size, EMBEDDING_DIM))

for word, i in word_index.items():
    if i < actual_vocab_size:  # Ensure index is within bounds
        if word in cbow_model.wv:
            embedding_matrix[i] = cbow_model.wv[word]

# --- Step 3: Define the BiLSTM + Attention model ---
inputs = Input(shape=(MAX_SEQUENCE_LENGTH,))
embedding_layer = Embedding(
    input_dim=actual_vocab_size,
    output_dim=EMBEDDING_DIM,
    weights=[embedding_matrix],
    input_length=MAX_SEQUENCE_LENGTH,
    trainable=False  # Set trainable=True to fine-tune embeddings
)(inputs)

bilstm_layer = Bidirectional(LSTM(units=LSTM_UNITS, return_sequences=True))(embedding_layer)

# Custom Attention Mechanism
attention_scores = Dense(1, activation="tanh")(bilstm_layer)
attention_weights = Dense(1, activation="softmax")(attention_scores)
context_vector = bilstm_layer * attention_weights

pooled_output = GlobalAveragePooling1D()(context_vector)
dropout_layer = Dropout(0.5)(pooled_output)
dense_layer = Dense(256, activation='relu')(dropout_layer)
dropout_layer2 = Dropout(0.5)(dense_layer)
output = Dense(num_classes, activation='softmax')(dropout_layer2)

model_bilstm_attention = Model(inputs, output)
model_bilstm_attention.compile(optimizer=Adam(clipnorm=1.0), loss='categorical_crossentropy', metrics=['accuracy'])
model_bilstm_attention.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3             │ (None, 100)            │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ embedding_3 (Embedding)   │ (None, 100, 300)       │      3,000,000 │ input_layer_3[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ bidirectional_2           │ (None, 100, 1024)      │      3,330,048 │ embedding_3[0][0]      │
│ (Bidirectional)           │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_5 (Dense)           │ (None, 100, 1)         │          1,025 │ bidirectional_2[0][0]  │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_6 (Dense)           │ (None, 100, 1)         │              2 │ dense_5[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ multiply (Multiply)       │ (None, 100, 1024)      │              0 │ bidirectional_2[0][0], │
│                           │                        │                │ dense_6[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ global_average_pooling1d  │ (None, 1024)           │              0 │ multiply[0][0]         │
│ (GlobalAveragePooling1D)  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout (Dropout)         │ (None, 1024)           │              0 │ global_average_poolin… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_7 (Dense)           │ (None, 256)            │        262,400 │ dropout[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_1 (Dropout)       │ (None, 256)            │              0 │ dense_7[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_8 (Dense)           │ (None, 3)              │            771 │ dropout_1[0][0]        │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 6,594,246 (25.16 MB)

 Trainable params: 3,594,246 (13.71 MB)

 Non-trainable params: 3,000,000 (11.44 MB)

In [ ]:


# --- Step 4: Train the model ---
early_stopping = EarlyStopping(monitor='val_loss', patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True)
history_bilstm_attention = model_bilstm_attention.fit(
    X_train_padded, y_train_categorical,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_data=(X_val_padded, y_val_categorical),
    verbose=1, callbacks=[early_stopping]
)

# --- Step 5: Evaluate on validation and test sets ---
val_loss, val_accuracy = model_bilstm_attention.evaluate(X_val_padded, y_val_categorical, verbose=0)
test_loss, test_accuracy = model_bilstm_attention.evaluate(X_test_padded, y_test_categorical, verbose=0)

print(f"\nCBOW + BiLSTM + Attention Validation Accuracy: {val_accuracy:.4f}")
print(f"CBOW + BiLSTM + Attention Test Accuracy: {test_accuracy:.4f}")

# --- Step 6: Predict on the test set and compute metrics ---
y_test_pred_probs = model_bilstm_attention.predict(X_test_padded, verbose=0)
y_test_pred_classes = np.argmax(y_test_pred_probs, axis=1)
y_test_true_classes = np.argmax(y_test_categorical, axis=1)

# Overall Metrics
overall_accuracy = accuracy_score(y_test_true_classes, y_test_pred_classes)
macro_roc_auc = roc_auc_score(y_test_categorical, y_test_pred_probs, average='macro', multi_class='ovr')

print("\n--- CBOW + BiLSTM + Attention Model Evaluation Metrics ---\n")

# Per-Class Metrics
for class_index, class_name in enumerate(label_encoder.classes_):
    # Binary labels for the current class
    y_true_binary = (y_test_true_classes == class_index).astype(int)
    y_pred_binary = (y_test_pred_classes == class_index).astype(int)

    # Calculate Metrics
    precision = precision_score(y_true_binary, y_pred_binary, zero_division=0)
    recall = recall_score(y_true_binary, y_pred_binary, zero_division=0)
    f1 = f1_score(y_true_binary, y_pred_binary, zero_division=0)
    roc_auc = roc_auc_score(y_true_binary, y_test_pred_probs[:, class_index])
    class_accuracy = accuracy_score(y_true_binary, y_pred_binary)

    # Print Metrics for the Current Class
    print(f"Class: {class_name}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1-Score: {f1:.4f}")
    print(f"  ROC-AUC: {roc_auc:.4f}")
    print(f"  Accuracy: {class_accuracy:.4f}\n")

# Overall Performance Metrics
print("Overall Performance:")
print(f"  Accuracy: {overall_accuracy:.4f}")
print(f"  Macro-Averaged ROC-AUC: {macro_roc_auc:.4f}")


Epoch 1/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 89s 187ms/step - accuracy: 0.8984 - loss: 0.2653 - val_accuracy: 0.8759 - val_loss: 0.3739
Epoch 2/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 93s 196ms/step - accuracy: 0.9381 - loss: 0.1706 - val_accuracy: 0.8677 - val_loss: 0.5028
Epoch 3/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 142s 197ms/step - accuracy: 0.9663 - loss: 0.0976 - val_accuracy: 0.8621 - val_loss: 0.5940
Epoch 4/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 142s 197ms/step - accuracy: 0.9773 - loss: 0.0696 - val_accuracy: 0.8633 - val_loss: 0.6422

CBOW + BiLSTM + Attention Validation Accuracy: 0.8759
CBOW + BiLSTM + Attention Test Accuracy: 0.8676

--- CBOW + BiLSTM + Attention Model Evaluation Metrics ---

Class: 0
  Precision: 0.8621
  Recall: 0.8357
  F1-Score: 0.8487
  ROC-AUC: 0.9684
  Accuracy: 0.9329

Class: 1
  Precision: 0.8005
  Recall: 0.8214
  F1-Score: 0.8108
  ROC-AUC: 0.9511
  Accuracy: 0.8990

Class: 2
  Precision: 0.9055
  Recall: 0.9055
  F1-Score: 0.9055
  ROC-AUC: 0.9629
  Accuracy: 0.9034

O

# **GRU**

In [ ]:
from tensorflow.keras.layers import GRU, Dense, Embedding, Dropout, Bidirectional
from tensorflow.keras.models import Sequential
from tensorflow.keras.regularizers import l2
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, accuracy_score
import numpy as np

# --- Configuration ---
EMBEDDING_DIM = 300  # Embedding dimensions
MAX_SEQUENCE_LENGTH = 100  # Maximum sequence length
VOCAB_SIZE = 10000  # Maximum vocabulary size
BATCH_SIZE = 256  # Batch size
EPOCHS = 20  # Number of epochs
EARLY_STOPPING_PATIENCE = 3  # Early stopping patience
GRU_UNITS = 512  # Number of GRU units

# --- Step 1: Define GRU Model ---
model_gru = Sequential([
    # Embedding Layer
    Embedding(
        input_dim=actual_vocab_size,
        output_dim=EMBEDDING_DIM,
        weights=[embedding_matrix],
        input_length=MAX_SEQUENCE_LENGTH,
        trainable=False  # Set to True if embeddings need fine-tuning
    ),

    # Bidirectional GRU Layer with L2 Regularization
    Bidirectional(
        GRU(units=GRU_UNITS, return_sequences=False, kernel_regularizer=l2(0.01))
    ),

    # Dropout Layer for Regularization
    Dropout(0.5),

    # Dense Layer with L2 Regularization
    Dense(64, activation='relu', kernel_regularizer=l2(0.01)),

    # Output Layer
    Dense(num_classes, activation='softmax')
])

# Compile the GRU Model
model_gru.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_gru.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)              │ ?                           │       3,000,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_3 (Bidirectional)      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_9 (Dense)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_10 (Dense)                     │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,000,000 (11.44 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 3,000,000 (11.44 MB)

In [ ]:


# --- Step 2: Train the GRU Model ---
history_gru = model_gru.fit(
    X_train_padded, y_train_categorical,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_data=(X_val_padded, y_val_categorical),
    verbose=1, callbacks=[early_stopping]
)



# --- Step 3: Evaluate the GRU Model ---

# Predict on Test Set
y_test_pred_probs = model_gru.predict(X_test_padded, verbose=0)  # Predicted probabilities
y_test_pred_classes = np.argmax(y_test_pred_probs, axis=1)  # Predicted class labels
y_test_true_classes = np.argmax(y_test_categorical, axis=1)  # True class labels

# Overall Metrics
overall_accuracy = accuracy_score(y_test_true_classes, y_test_pred_classes)
macro_roc_auc = roc_auc_score(y_test_categorical, y_test_pred_probs, average='macro', multi_class='ovr')

print("\n--- GRU Model Evaluation Metrics ---\n")

# Per-Class Metrics
for class_index, class_name in enumerate(label_encoder.classes_):
    # Binary labels for the current class
    y_true_binary = (y_test_true_classes == class_index).astype(int)
    y_pred_binary = (y_test_pred_classes == class_index).astype(int)

    # Calculate Metrics
    precision = precision_score(y_true_binary, y_pred_binary, zero_division=0)
    recall = recall_score(y_true_binary, y_pred_binary, zero_division=0)
    f1 = f1_score(y_true_binary, y_pred_binary, zero_division=0)
    roc_auc = roc_auc_score(y_true_binary, y_test_pred_probs[:, class_index])
    class_accuracy = accuracy_score(y_true_binary, y_pred_binary)

    # Print Metrics for the Current Class
    print(f"Class: {class_name}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1-Score: {f1:.4f}")
    print(f"  ROC-AUC: {roc_auc:.4f}")
    print(f"  Accuracy: {class_accuracy:.4f}\n")

# Overall Performance Metrics
print("Overall Performance:")
print(f"  Accuracy: {overall_accuracy:.4f}")
print(f"  Macro-Averaged ROC-AUC: {macro_roc_auc:.4f}")

# --- Validation and Test Evaluation ---
val_loss, val_accuracy = model_gru.evaluate(X_val_padded, y_val_categorical, verbose=0)
print(f"\nGRU Validation Accuracy: {val_accuracy:.4f}")

test_loss, test_accuracy = model_gru.evaluate(X_test_padded, y_test_categorical, verbose=0)
print(f"GRU Test Accuracy: {test_accuracy:.4f}")


Epoch 1/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 71s 141ms/step - accuracy: 0.7904 - loss: 2.3906 - val_accuracy: 0.8621 - val_loss: 0.4617
Epoch 2/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 84s 145ms/step - accuracy: 0.8439 - loss: 0.4758 - val_accuracy: 0.8558 - val_loss: 0.4552
Epoch 3/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 82s 146ms/step - accuracy: 0.8528 - loss: 0.4572 - val_accuracy: 0.8621 - val_loss: 0.4569

--- GRU Model Evaluation Metrics ---

Class: 0
  Precision: 0.8412
  Recall: 0.7967
  F1-Score: 0.8183
  ROC-AUC: 0.9553
  Accuracy: 0.9203

Class: 1
  Precision: 0.7645
  Recall: 0.8500
  F1-Score: 0.8050
  ROC-AUC: 0.9506
  Accuracy: 0.8915

Class: 2
  Precision: 0.9123
  Recall: 0.8810
  F1-Score: 0.8964
  ROC-AUC: 0.9617
  Accuracy: 0.8959

Overall Performance:
  Accuracy: 0.8538
  Macro-Averaged ROC-AUC: 0.9559

GRU Validation Accuracy: 0.8621
GRU Test Accuracy: 0.8538


# **GRU With CBOW**

In [ ]:
from gensim.models import Word2Vec
from tensorflow.keras.layers import GRU, Dense, Embedding, Dropout, Bidirectional
from tensorflow.keras.models import Sequential
from tensorflow.keras.regularizers import l2
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, accuracy_score
import numpy as np

# --- Configuration ---
EMBEDDING_DIM = 300  # Dimension of Word2Vec embeddings
GRU_UNITS = 512  # Number of GRU units
MAX_SEQUENCE_LENGTH = 100  # Maximum sequence length
VOCAB_SIZE = 10000  # Maximum vocabulary size
BATCH_SIZE =  256 # Batch size
EPOCHS = 50  # Number of epochs
EARLY_STOPPING_PATIENCE = 3  # Patience for early stopping

# --- Step 1: Train CBOW Word2Vec model ---
# Assuming `tokenized_sentences` is a list of tokenized sentences from `X_train`
cbow_model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=EMBEDDING_DIM,
    window=5,
    min_count=1,
    workers=4,
    sg=0  # CBOW model
)

# --- Step 2: Build the embedding matrix ---
word_index = tokenizer.word_index
actual_vocab_size = min(len(word_index) + 1, VOCAB_SIZE)  # Limit vocab size to VOCAB_SIZE
embedding_matrix = np.zeros((actual_vocab_size, EMBEDDING_DIM))

for word, i in word_index.items():
    if i < actual_vocab_size:  # Ensure index is within bounds
        if word in cbow_model.wv:
            embedding_matrix[i] = cbow_model.wv[word]

# --- Step 3: Define GRU model with CBOW embeddings ---
model_gru_cbow = Sequential([
    Embedding(
        input_dim=actual_vocab_size,
        output_dim=EMBEDDING_DIM,
        weights=[embedding_matrix],
        input_length=MAX_SEQUENCE_LENGTH,
        trainable=False  # Non-trainable CBOW embeddings
    ),

    # Bidirectional GRU layer with L2 regularization
    Bidirectional(GRU(units=GRU_UNITS, return_sequences=False, kernel_regularizer=l2(0.01))),
    Dropout(0.5),  # Regularization

    # Dense layer with L2 regularization
    Dense(512, activation='relu', kernel_regularizer=l2(0.01)),

    # Output layer
    Dense(num_classes, activation='softmax')
])

# --- Step 4: Compile the model ---
model_gru_cbow.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Display model summary
model_gru_cbow.summary()

# --- Step 5: Train the model ---
early_stopping = EarlyStopping(monitor='val_loss', patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True)
history_gru_cbow = model_gru_cbow.fit(
    X_train_padded, y_train_categorical,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_data=(X_val_padded, y_val_categorical),
    verbose=1, callbacks=[early_stopping]
)

# --- Step 6: Evaluate the model ---
# Evaluate on validation set
val_loss, val_accuracy = model_gru_cbow.evaluate(X_val_padded, y_val_categorical, verbose=0)
print(f"\nCBOW + GRU Validation Accuracy: {val_accuracy:.4f}")

# Evaluate on test set
test_loss, test_accuracy = model_gru_cbow.evaluate(X_test_padded, y_test_categorical, verbose=0)
print(f"CBOW + GRU Test Accuracy: {test_accuracy:.4f}")

# --- Step 7: Predict and Compute Metrics ---
y_test_pred_probs = model_gru_cbow.predict(X_test_padded, verbose=0)  # Predicted probabilities
y_test_pred_classes = np.argmax(y_test_pred_probs, axis=1)  # Predicted class labels
y_test_true_classes = np.argmax(y_test_categorical, axis=1)  # True class labels

# Overall Metrics
overall_accuracy = accuracy_score(y_test_true_classes, y_test_pred_classes)
macro_roc_auc = roc_auc_score(y_test_categorical, y_test_pred_probs, average='macro', multi_class='ovr')

print("\n--- CBOW + GRU Model Evaluation Metrics ---\n")

# Per-Class Metrics
for class_index, class_name in enumerate(label_encoder.classes_):
    # Binary labels for the current class
    y_true_binary = (y_test_true_classes == class_index).astype(int)
    y_pred_binary = (y_test_pred_classes == class_index).astype(int)

    # Calculate Metrics
    precision = precision_score(y_true_binary, y_pred_binary, zero_division=0)
    recall = recall_score(y_true_binary, y_pred_binary, zero_division=0)
    f1 = f1_score(y_true_binary, y_pred_binary, zero_division=0)
    roc_auc = roc_auc_score(y_true_binary, y_test_pred_probs[:, class_index])
    class_accuracy = accuracy_score(y_true_binary, y_pred_binary)

    # Print Metrics for the Current Class
    print(f"Class: {class_name}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1-Score: {f1:.4f}")
    print(f"  ROC-AUC: {roc_auc:.4f}")
    print(f"  Accuracy: {class_accuracy:.4f}\n")

# Overall Performance Metrics
print("Overall Performance:")
print(f"  Accuracy: {overall_accuracy:.4f}")
print(f"  Macro-Averaged ROC-AUC: {macro_roc_auc:.4f}")


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)              │ ?                           │       3,000,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_4 (Bidirectional)      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_11 (Dense)                     │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_12 (Dense)                     │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,000,000 (11.44 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 3,000,000 (11.44 MB)

Epoch 1/50
475/475 ━━━━━━━━━━━━━━━━━━━━ 73s 148ms/step - accuracy: 0.7849 - loss: 3.2940 - val_accuracy: 0.8514 - val_loss: 0.4940
Epoch 2/50
475/475 ━━━━━━━━━━━━━━━━━━━━ 81s 147ms/step - accuracy: 0.8417 - loss: 0.4851 - val_accuracy: 0.8633 - val_loss: 0.4707
Epoch 3/50
475/475 ━━━━━━━━━━━━━━━━━━━━ 82s 147ms/step - accuracy: 0.8533 - loss: 0.4629 - val_accuracy: 0.8583 - val_loss: 0.4631
Epoch 4/50
475/475 ━━━━━━━━━━━━━━━━━━━━ 69s 145ms/step - accuracy: 0.8600 - loss: 0.4548 - val_accuracy: 0.8395 - val_loss: 0.5161
Epoch 5/50
475/475 ━━━━━━━━━━━━━━━━━━━━ 83s 147ms/step - accuracy: 0.8664 - loss: 0.4440 - val_accuracy: 0.8577 - val_loss: 0.4784
Epoch 6/50
475/475 ━━━━━━━━━━━━━━━━━━━━ 82s 147ms/step - accuracy: 0.8749 - loss: 0.4310 - val_accuracy: 0.8364 - val_loss: 0.5447

CBOW + GRU Validation Accuracy: 0.8583
CBOW + GRU Test Accuracy: 0.8588

--- CBOW + GRU Model Evaluation Metrics ---

Class: 0
  Precision: 0.8774
  Recall: 0.7577
  F1-Score: 0.8132
  ROC-AUC: 0.9540
  Accuracy: 

# **GRU With Attention**

In [ ]:
from tensorflow.keras.layers import GRU, Dense, Embedding, Input, GlobalAveragePooling1D, Add, Attention, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, accuracy_score
import numpy as np

# --- Configuration ---
EMBEDDING_DIM = 300  # Dimension of word embeddings
GRU_UNITS = 512  # Number of GRU units
MAX_SEQUENCE_LENGTH = 100  # Maximum sequence length
VOCAB_SIZE = 10000  # Maximum vocabulary size
BATCH_SIZE = 256  # Batch size
EPOCHS = 20  # Number of epochs
EARLY_STOPPING_PATIENCE = 3  # Early stopping patience

# --- Step 1: Define the GRU model with attention ---
input_layer = Input(shape=(MAX_SEQUENCE_LENGTH,))

# Embedding layer with pre-trained word embeddings
embedding_layer = Embedding(
    input_dim=actual_vocab_size,
    output_dim=EMBEDDING_DIM,
    weights=[embedding_matrix],
    input_length=MAX_SEQUENCE_LENGTH,
    trainable=False  # Non-trainable embeddings
)(input_layer)

# GRU layer
gru_layer = GRU(units=GRU_UNITS, return_sequences=True)(embedding_layer)

# Attention layer
attention_layer = Attention(use_scale=True)([gru_layer, gru_layer])

# Add attention output to the GRU outputs
attention_output = Add()([gru_layer, attention_layer])

# Apply global average pooling to reduce the sequence dimension
pooled_output = GlobalAveragePooling1D()(attention_output)

# Dropout layer for regularization
dropout_layer = Dropout(0.5)(pooled_output)

# Fully connected Dense layers
dense_output = Dense(256, activation='relu')(dropout_layer)
output_layer = Dense(num_classes, activation='softmax')(dense_output)

# Define the model
model_gru_attention = Model(inputs=input_layer, outputs=output_layer)

# --- Step 2: Compile the model ---
model_gru_attention.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_gru_attention.summary()

# --- Step 3: Train the model ---
early_stopping = EarlyStopping(monitor='val_loss', patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True)
history_gru_attention = model_gru_attention.fit(
    X_train_padded, y_train_categorical,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_data=(X_val_padded, y_val_categorical),
    verbose=1, callbacks=[early_stopping]
)

# --- Step 4: Evaluate the model ---
# Evaluate on validation set
val_loss, val_accuracy = model_gru_attention.evaluate(X_val_padded, y_val_categorical, verbose=0)
print(f"\nGRU with Attention Validation Accuracy: {val_accuracy:.4f}")

# Evaluate on test set
test_loss, test_accuracy = model_gru_attention.evaluate(X_test_padded, y_test_categorical, verbose=0)
print(f"GRU with Attention Test Accuracy: {test_accuracy:.4f}")

# --- Step 5: Predict and Compute Metrics ---
y_test_pred_probs = model_gru_attention.predict(X_test_padded, verbose=0)  # Predicted probabilities
y_test_pred_classes = np.argmax(y_test_pred_probs, axis=1)  # Predicted class labels
y_test_true_classes = np.argmax(y_test_categorical, axis=1)  # True class labels

# Overall Metrics
overall_accuracy = accuracy_score(y_test_true_classes, y_test_pred_classes)
macro_roc_auc = roc_auc_score(y_test_categorical, y_test_pred_probs, average='macro', multi_class='ovr')

print("\n--- GRU with Attention Model Evaluation Metrics ---\n")

# Per-Class Metrics
for class_index, class_name in enumerate(label_encoder.classes_):
    # Binary labels for the current class
    y_true_binary = (y_test_true_classes == class_index).astype(int)
    y_pred_binary = (y_test_pred_classes == class_index).astype(int)

    # Calculate Metrics
    precision = precision_score(y_true_binary, y_pred_binary, zero_division=0)
    recall = recall_score(y_true_binary, y_pred_binary, zero_division=0)
    f1 = f1_score(y_true_binary, y_pred_binary, zero_division=0)
    roc_auc = roc_auc_score(y_true_binary, y_test_pred_probs[:, class_index])
    class_accuracy = accuracy_score(y_true_binary, y_pred_binary)

    # Print Metrics for the Current Class
    print(f"Class: {class_name}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1-Score: {f1:.4f}")
    print(f"  ROC-AUC: {roc_auc:.4f}")
    print(f"  Accuracy: {class_accuracy:.4f}\n")

# Overall Performance Metrics
print("Overall Performance:")
print(f"  Accuracy: {overall_accuracy:.4f}")
print(f"  Macro-Averaged ROC-AUC: {macro_roc_auc:.4f}")


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6             │ (None, 100)            │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ embedding_6 (Embedding)   │ (None, 100, 300)       │      3,000,000 │ input_layer_6[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ gru_2 (GRU)               │ (None, 100, 512)       │      1,250,304 │ embedding_6[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ attention (Attention)     │ (None, 100, 512)       │              1 │ gru_2[0][0],           │
│                           │                        │                │ gru_2[0][0]            │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add (Add)                 │ (None, 100, 512)       │              0 │ gru_2[0][0],           │
│                           │                        │                │ attention[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ global_average_pooling1d… │ (None, 512)            │              0 │ add[0][0]              │
│ (GlobalAveragePooling1D)  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_4 (Dropout)       │ (None, 512)            │              0 │ global_average_poolin… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_13 (Dense)          │ (None, 256)            │        131,328 │ dropout_4[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_14 (Dense)          │ (None, 3)              │            771 │ dense_13[0][0]         │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 4,382,404 (16.72 MB)

 Trainable params: 1,382,404 (5.27 MB)

 Non-trainable params: 3,000,000 (11.44 MB)

Epoch 1/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 42s 85ms/step - accuracy: 0.8179 - loss: 0.4715 - val_accuracy: 0.8658 - val_loss: 0.3578
Epoch 2/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 40s 84ms/step - accuracy: 0.8948 - loss: 0.2697 - val_accuracy: 0.8784 - val_loss: 0.4253
Epoch 3/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 41s 84ms/step - accuracy: 0.9367 - loss: 0.1702 - val_accuracy: 0.8614 - val_loss: 0.5188
Epoch 4/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 41s 84ms/step - accuracy: 0.9577 - loss: 0.1158 - val_accuracy: 0.8489 - val_loss: 0.7112

GRU with Attention Validation Accuracy: 0.8658
GRU with Attention Test Accuracy: 0.8651

--- GRU with Attention Model Evaluation Metrics ---

Class: 0
  Precision: 0.8362
  Recall: 0.8245
  F1-Score: 0.8303
  ROC-AUC: 0.9663
  Accuracy: 0.9241

Class: 1
  Precision: 0.8065
  Recall: 0.8333
  F1-Score: 0.8197
  ROC-AUC: 0.9583
  Accuracy: 0.9034

Class: 2
  Precision: 0.9094
  Recall: 0.8994
  F1-Score: 0.9044
  ROC-AUC: 0.9701
  Accuracy: 0.9028

Overall Performance:
  Accur

# **GRU With CBOW + Attention**

In [ ]:
from gensim.models import Word2Vec
from tensorflow.keras.layers import GRU, Dense, Embedding, Input, GlobalAveragePooling1D, Add, Attention, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, accuracy_score
import numpy as np

# --- Configuration ---
EMBEDDING_DIM = 300  # Dimension of word embeddings
GRU_UNITS = 512  # Number of GRU units
MAX_SEQUENCE_LENGTH = 100  # Maximum sequence length
VOCAB_SIZE = 10000  # Maximum vocabulary size
BATCH_SIZE =  256 # Batch size
EPOCHS = 20  # Number of epochs
EARLY_STOPPING_PATIENCE = 3  # Early stopping patience

# --- Step 1: Train CBOW Word2Vec model ---
# Assuming `tokenized_sentences` is a list of tokenized sentences from `X_train`
cbow_model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=EMBEDDING_DIM,
    window=5,
    min_count=1,
    workers=4,
    sg=0  # CBOW model
)

# --- Step 2: Build the embedding matrix ---
word_index = tokenizer.word_index
actual_vocab_size = min(len(word_index) + 1, VOCAB_SIZE)  # Limit vocab size to VOCAB_SIZE
embedding_matrix = np.zeros((actual_vocab_size, EMBEDDING_DIM))

for word, i in word_index.items():
    if i < actual_vocab_size:  # Ensure index is within bounds
        if word in cbow_model.wv:
            embedding_matrix[i] = cbow_model.wv[word]

# --- Step 3: Define the GRU model with CBOW embeddings and Attention ---
input_layer = Input(shape=(MAX_SEQUENCE_LENGTH,))

# Embedding layer with CBOW embeddings
embedding_layer = Embedding(
    input_dim=actual_vocab_size,
    output_dim=EMBEDDING_DIM,
    weights=[embedding_matrix],
    input_length=MAX_SEQUENCE_LENGTH,
    trainable=False  # Non-trainable embeddings
)(input_layer)

# GRU layer
gru_layer = GRU(units=GRU_UNITS, return_sequences=True)(embedding_layer)

# Attention layer
attention_layer = Attention(use_scale=True)([gru_layer, gru_layer])

# Add the GRU outputs and attention outputs
attention_output = Add()([gru_layer, attention_layer])

# Apply global average pooling to reduce sequence dimension
pooled_output = GlobalAveragePooling1D()(attention_output)

# Dropout for regularization
dropout_layer = Dropout(0.5)(pooled_output)

# Dense layers
dense_output = Dense(256, activation='relu')(dropout_layer)
output_layer = Dense(num_classes, activation='softmax')(dense_output)

# Define the model
model_gru_attention_cbow = Model(inputs=input_layer, outputs=output_layer)

# --- Step 4: Compile the model ---
model_gru_attention_cbow.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_gru_attention_cbow.summary()

# --- Step 5: Early stopping callback ---
early_stopping = EarlyStopping(monitor='val_loss', patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True)

# --- Step 6: Train the model ---
history_gru_attention_cbow = model_gru_attention_cbow.fit(
    X_train_padded, y_train_categorical,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_data=(X_val_padded, y_val_categorical),
    verbose=1, callbacks=[early_stopping]
)

# --- Step 7: Evaluate the model on validation and test sets ---
# Validation set
val_loss, val_accuracy = model_gru_attention_cbow.evaluate(X_val_padded, y_val_categorical, verbose=0)
print(f"\nCBOW + GRU with Attention Validation Accuracy: {val_accuracy:.4f}")

# Test set
test_loss, test_accuracy = model_gru_attention_cbow.evaluate(X_test_padded, y_test_categorical, verbose=0)
print(f"CBOW + GRU with Attention Test Accuracy: {test_accuracy:.4f}")

# --- Step 8: Predict and Compute Metrics ---
y_test_pred_probs = model_gru_attention_cbow.predict(X_test_padded, verbose=0)  # Predicted probabilities
y_test_pred_classes = np.argmax(y_test_pred_probs, axis=1)  # Predicted class labels
y_test_true_classes = np.argmax(y_test_categorical, axis=1)  # True class labels

# Overall Metrics
overall_accuracy = accuracy_score(y_test_true_classes, y_test_pred_classes)
macro_roc_auc = roc_auc_score(y_test_categorical, y_test_pred_probs, average='macro', multi_class='ovr')

print("\n--- CBOW + GRU with Attention Model Evaluation Metrics ---\n")

# Per-Class Metrics
for class_index, class_name in enumerate(label_encoder.classes_):
    # Binary labels for the current class
    y_true_binary = (y_test_true_classes == class_index).astype(int)
    y_pred_binary = (y_test_pred_classes == class_index).astype(int)

    # Calculate Metrics
    precision = precision_score(y_true_binary, y_pred_binary, zero_division=0)
    recall = recall_score(y_true_binary, y_pred_binary, zero_division=0)
    f1 = f1_score(y_true_binary, y_pred_binary, zero_division=0)
    roc_auc = roc_auc_score(y_true_binary, y_test_pred_probs[:, class_index])
    class_accuracy = accuracy_score(y_true_binary, y_pred_binary)

    # Print Metrics for the Current Class
    print(f"Class: {class_name}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1-Score: {f1:.4f}")
    print(f"  ROC-AUC: {roc_auc:.4f}")
    print(f"  Accuracy: {class_accuracy:.4f}\n")

# Overall Performance Metrics
print("Overall Performance:")
print(f"  Accuracy: {overall_accuracy:.4f}")
print(f"  Macro-Averaged ROC-AUC: {macro_roc_auc:.4f}")


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_7             │ (None, 100)            │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ embedding_7 (Embedding)   │ (None, 100, 300)       │      3,000,000 │ input_layer_7[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ gru_3 (GRU)               │ (None, 100, 512)       │      1,250,304 │ embedding_7[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ attention_1 (Attention)   │ (None, 100, 512)       │              1 │ gru_3[0][0],           │
│                           │                        │                │ gru_3[0][0]            │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_1 (Add)               │ (None, 100, 512)       │              0 │ gru_3[0][0],           │
│                           │                        │                │ attention_1[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ global_average_pooling1d… │ (None, 512)            │              0 │ add_1[0][0]            │
│ (GlobalAveragePooling1D)  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_5 (Dropout)       │ (None, 512)            │              0 │ global_average_poolin… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_15 (Dense)          │ (None, 256)            │        131,328 │ dropout_5[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_16 (Dense)          │ (None, 3)              │            771 │ dense_15[0][0]         │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 4,382,404 (16.72 MB)

 Trainable params: 1,382,404 (5.27 MB)

 Non-trainable params: 3,000,000 (11.44 MB)

Epoch 1/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 41s 81ms/step - accuracy: 0.8135 - loss: 0.4784 - val_accuracy: 0.8740 - val_loss: 0.3655
Epoch 2/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 42s 83ms/step - accuracy: 0.8933 - loss: 0.2753 - val_accuracy: 0.8740 - val_loss: 0.4238
Epoch 3/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 42s 85ms/step - accuracy: 0.9366 - loss: 0.1685 - val_accuracy: 0.8677 - val_loss: 0.5651
Epoch 4/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 41s 84ms/step - accuracy: 0.9570 - loss: 0.1192 - val_accuracy: 0.8583 - val_loss: 0.6103

CBOW + GRU with Attention Validation Accuracy: 0.8740
CBOW + GRU with Attention Test Accuracy: 0.8708

--- CBOW + GRU with Attention Model Evaluation Metrics ---

Class: 0
  Precision: 0.8338
  Recall: 0.8384
  F1-Score: 0.8361
  ROC-AUC: 0.9700
  Accuracy: 0.9260

Class: 1
  Precision: 0.8265
  Recall: 0.8167
  F1-Score: 0.8216
  ROC-AUC: 0.9561
  Accuracy: 0.9065

Class: 2
  Precision: 0.9095
  Recall: 0.9129
  F1-Score: 0.9112
  ROC-AUC: 0.9677
  Accuracy: 0.9090

Overall

# **Hybrid Model**

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, Conv1D, MaxPooling1D, LSTM, Dense,
    Flatten, Dropout, Attention, Bidirectional, Concatenate, GlobalAveragePooling1D
)
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, accuracy_score
import numpy as np
from gensim.models import Word2Vec
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

# --- Configuration ---
EMBEDDING_DIM = 300  # Dimension of word embeddings
MAX_SEQUENCE_LENGTH = 100  # Maximum sequence length
VOCAB_SIZE = 10000  # Maximum vocabulary size
BATCH_SIZE = 256  # Batch size
EPOCHS = 20  # Number of epochs
LEARNING_RATE = 1e-4  # Learning rate for optimizer
EARLY_STOPPING_PATIENCE = 3  # Patience for early stopping
CNN_FILTERS = 128  # Number of CNN filters
LSTM_UNITS = 512  # Number of LSTM units

# --- Step 1: Train a Word2Vec model ---
# Assuming `tokenized_sentences` is a list of tokenized sentences from `X_train`
word2vec_model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=EMBEDDING_DIM,
    window=5,
    min_count=1,
    workers=4
)

# --- Step 2: Build the embedding matrix ---
word_index = tokenizer.word_index
actual_vocab_size = min(len(word_index) + 1, VOCAB_SIZE)  # Limit vocab size
embedding_matrix = np.zeros((actual_vocab_size, EMBEDDING_DIM))

for word, i in word_index.items():
    if i < actual_vocab_size:  # Ensure index is within bounds
        if word in word2vec_model.wv:
            embedding_matrix[i] = word2vec_model.wv[word]

# --- Step 3: Convert labels to categorical (one-hot encoding) ---
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

num_classes = len(label_encoder.classes_)
y_train_categorical = to_categorical(y_train_encoded, num_classes=num_classes)
y_val_categorical = to_categorical(y_val_encoded, num_classes=num_classes)
y_test_categorical = to_categorical(y_test_encoded, num_classes=num_classes)

# --- Step 4: Define Hybrid CNN + RNN with Attention Model ---
input_layer = Input(shape=(MAX_SEQUENCE_LENGTH,))

# Embedding Layer
embedding_layer = Embedding(
    input_dim=actual_vocab_size,
    output_dim=EMBEDDING_DIM,
    weights=[embedding_matrix],
    input_length=MAX_SEQUENCE_LENGTH,
    trainable=False  # Non-trainable embeddings
)(input_layer)

# CNN Block
cnn = Conv1D(filters=CNN_FILTERS, kernel_size=3, activation='relu', padding='same')(embedding_layer)
cnn = MaxPooling1D(pool_size=2)(cnn)

# RNN Block
rnn = Bidirectional(LSTM(units=LSTM_UNITS, return_sequences=True))(cnn)
rnn = Bidirectional(LSTM(units=LSTM_UNITS, return_sequences=True))(rnn)

# Attention Mechanism
query = Dense(64, activation="relu")(rnn)  # Query vector
key = Dense(64, activation="relu")(rnn)   # Key vector
value = Dense(64, activation="relu")(rnn)  # Value vector

attention_layer = Attention()([query, key, value])
attention_output = GlobalAveragePooling1D()(attention_layer)  # Pooling attention outputs

# Fully Connected Layers
fc = Dense(128, activation="relu")(attention_output)
fc = Dropout(0.5)(fc)
output_layer = Dense(num_classes, activation="softmax")(fc)

# --- Step 5: Compile the model ---
model_hybrid = Model(inputs=input_layer, outputs=output_layer)
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
model_hybrid.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Display model summary
model_hybrid.summary()

# --- Step 6: Train the model ---
early_stopping = EarlyStopping(monitor='val_loss', patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True)
history = model_hybrid.fit(
    X_train_padded, y_train_categorical,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_data=(X_val_padded, y_val_categorical),
    verbose=1, callbacks=[early_stopping]
)

# --- Step 7: Evaluate the model on validation and test sets ---
# Validation set
val_loss, val_accuracy = model_hybrid.evaluate(X_val_padded, y_val_categorical, verbose=0)
print(f"\nHybrid Model Validation Accuracy: {val_accuracy:.4f}")

# Test set
test_loss, test_accuracy = model_hybrid.evaluate(X_test_padded, y_test_categorical, verbose=0)
print(f"Hybrid Model Test Accuracy: {test_accuracy:.4f}")

# --- Step 8: Predict and Compute Metrics ---
y_test_pred_probs = model_hybrid.predict(X_test_padded, verbose=0)  # Predicted probabilities
y_test_pred_classes = np.argmax(y_test_pred_probs, axis=1)  # Predicted class labels
y_test_true_classes = np.argmax(y_test_categorical, axis=1)  # True class labels

# Overall Metrics
overall_accuracy = accuracy_score(y_test_true_classes, y_test_pred_classes)
macro_roc_auc = roc_auc_score(y_test_categorical, y_test_pred_probs, average='macro', multi_class='ovr')

print("\n--- Hybrid CNN + RNN + Attention Model Evaluation Metrics ---\n")

# Per-Class Metrics
for class_index, class_name in enumerate(label_encoder.classes_):
    y_true_binary = (y_test_true_classes == class_index).astype(int)
    y_pred_binary = (y_test_pred_classes == class_index).astype(int)

    precision = precision_score(y_true_binary, y_pred_binary, zero_division=0)
    recall = recall_score(y_true_binary, y_pred_binary, zero_division=0)
    f1 = f1_score(y_true_binary, y_pred_binary, zero_division=0)
    roc_auc = roc_auc_score(y_true_binary, y_test_pred_probs[:, class_index])
    class_accuracy = accuracy_score(y_true_binary, y_pred_binary)

    print(f"Class: {class_name}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1-Score: {f1:.4f}")
    print(f"  ROC-AUC: {roc_auc:.4f}")
    print(f"  Accuracy: {class_accuracy:.4f}\n")

# Overall Performance Metrics
print("Overall Performance:")
print(f"  Accuracy: {overall_accuracy:.4f}")
print(f"  Macro-Averaged ROC-AUC: {macro_roc_auc:.4f}")


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "functional_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_9             │ (None, 100)            │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ embedding_9 (Embedding)   │ (None, 100, 300)       │      3,000,000 │ input_layer_9[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_1 (Conv1D)         │ (None, 100, 128)       │        115,328 │ embedding_9[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ max_pooling1d_1           │ (None, 50, 128)        │              0 │ conv1d_1[0][0]         │
│ (MaxPooling1D)            │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ bidirectional_7           │ (None, 50, 1024)       │      2,625,536 │ max_pooling1d_1[0][0]  │
│ (Bidirectional)           │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ bidirectional_8           │ (None, 50, 1024)       │      6,295,552 │ bidirectional_7[0][0]  │
│ (Bidirectional)           │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_22 (Dense)          │ (None, 50, 64)         │         65,600 │ bidirectional_8[0][0]  │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_23 (Dense)          │ (None, 50, 64)         │         65,600 │ bidirectional_8[0][0]  │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_24 (Dense)          │ (None, 50, 64)         │         65,600 │ bidirectional_8[0][0]  │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ attention_3 (Attention)   │ (None, 50, 64)         │              0 │ dense_22[0][0],        │
│                           │                        │                │ dense_23[0][0],        │
│                           │                        │                │ dense_24[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ global_average_pooling1d… │ (None, 64)             │              0 │ attention_3[0][0]      │
│ (GlobalAveragePooling1D)  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_25 (Dense)          │ (None, 128)            │          8,320 │ global_average_poolin… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_7 (Dropout)       │ (None, 128)            │              0 │ dense_25[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_26 (Dense)          │ (None, 3)              │            387 │ dropout_7[0][0]        │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 12,241,923 (46.70 MB)

 Trainable params: 9,241,923 (35.26 MB)

 Non-trainable params: 3,000,000 (11.44 MB)

Epoch 1/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 122s 245ms/step - accuracy: 0.7510 - loss: 0.6088 - val_accuracy: 0.8621 - val_loss: 0.3830
Epoch 2/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 145s 252ms/step - accuracy: 0.8591 - loss: 0.3706 - val_accuracy: 0.8621 - val_loss: 0.3646
Epoch 3/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 142s 252ms/step - accuracy: 0.8802 - loss: 0.3164 - val_accuracy: 0.8627 - val_loss: 0.3840
Epoch 4/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 142s 252ms/step - accuracy: 0.8939 - loss: 0.2808 - val_accuracy: 0.8683 - val_loss: 0.4139
Epoch 5/20
475/475 ━━━━━━━━━━━━━━━━━━━━ 120s 252ms/step - accuracy: 0.9147 - loss: 0.2294 - val_accuracy: 0.8596 - val_loss: 0.4420

Hybrid Model Validation Accuracy: 0.8621
Hybrid Model Test Accuracy: 0.8657

--- Hybrid CNN + RNN + Attention Model Evaluation Metrics ---

Class: 0
  Precision: 0.8293
  Recall: 0.8524
  F1-Score: 0.8407
  ROC-AUC: 0.9632
  Accuracy: 0.9272

Class: 1
  Precision: 0.8074
  Recall: 0.8286
  F1-Score: 0.8179
  ROC-AUC: 0.9507
  Accuracy: 0

# **Transformers**

In [ ]:
!pip install transformers datasets torch scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 16.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [ ]:
import torch
from transformers import BertForSequenceClassification, BertTokenizer
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
import pandas as pd
from transformers import RobertaForSequenceClassification, RobertaTokenizer

device = torch.device("cpu")


df['Label'] -= 1

# Load the model and tokenizer
model_name = 'roberta-base'
num_classes = 4
model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=num_classes)
tokenizer = RobertaTokenizer.from_pretrained(model_name)

# Split data into train, validation, and test sets
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Separate features and labels for each split
X_train, y_train = train_df['Review'], train_df['Label']
X_val, y_val = val_df['Review'], val_df['Label']
X_test, y_test = test_df['Review'], test_df['Label']

# Convert to strings and handle missing values
X_train = X_train.astype(str).fillna('')
X_val = X_val.astype(str).fillna('')
X_test = X_test.astype(str).fillna('')

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def preprocess_data(texts, labels, max_length):
    encodings = tokenizer(texts.tolist(), truncation=True, padding='max_length', max_length=max_length, return_tensors='pt')
    labels = torch.tensor(labels.tolist()).to(device)
    dataset = TensorDataset(encodings.input_ids.to(device), encodings.attention_mask.to(device), labels)
    return dataset

max_length = 512  # Set your desired max sequence length

train_dataset = preprocess_data(X_train, y_train, max_length)
val_dataset = preprocess_data(X_val, y_val, max_length)
test_dataset = preprocess_data(X_test, y_test, max_length)

batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

# Define optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

In [ ]:
# Model training
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

num_epochs = 20
early_stopping_patience = 3
best_val_loss = float('inf')
epochs_since_last_improvement = 0

for epoch in range(num_epochs):
    model.train()
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids, attention_mask, labels = batch
        input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids, attention_mask, labels = batch
            input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            val_loss += loss.item()
            predicted_labels = torch.argmax(outputs.logits, dim=1)
            correct += (predicted_labels == labels).sum().item()
            total += labels.size(0)

    val_accuracy = correct / total
    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Validation Accuracy: {val_accuracy:.4f}")
    print(f"Avg. Validation Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_since_last_improvement = 0
    else:
        epochs_since_last_improvement += 1
        if epochs_since_last_improvement >= early_stopping_patience:
            print("Early stopping triggered. Stopping training.")
            break

In [ ]:





# BERT model and tokenizer initialization
model_name = 'bert-base-multilingual-cased'
num_classes = 4
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=num_classes)
tokenizer = BertTokenizer.from_pretrained(model_name)

# Data preprocessing
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)


X_train, y_train = train_df['Review'], train_df['Label']
X_val, y_val = val_df['Review'], val_df['Label']
X_test, y_test = test_df['Review'], test_df['Label']

X_train = X_train.astype(str).fillna('')
X_val = X_val.astype(str).fillna('')
X_test = X_test.astype(str).fillna('')

def preprocess_data(texts, labels, max_length):
    encodings = tokenizer(texts.tolist(), truncation=True, padding='max_length', max_length=max_length, return_tensors='pt')
    labels = torch.tensor(labels.tolist())
    dataset = TensorDataset(encodings.input_ids, encodings.attention_mask, labels)
    return dataset

max_length = 512
train_dataset = preprocess_data(X_train, y_train, max_length)
val_dataset = preprocess_data(X_val, y_val, max_length)
test_dataset = preprocess_data(X_test, y_test, max_length)

batch_size = 8

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import torch
from transformers import BertForSequenceClassification, BertTokenizer
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load BERT model and tokenizer
model_name = 'bert-base-uncased'  # Change model name if needed
num_classes = 4  # Update based on your dataset
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=num_classes)
tokenizer = BertTokenizer.from_pretrained(model_name)

# Preprocess labels (if labels need adjustment)
df['Label'] -= 1

# Split data into train, validation, and test sets
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

X_train, y_train = train_df['Review'], train_df['Label']
X_val, y_val = val_df['Review'], val_df['Label']
X_test, y_test = test_df['Review'], test_df['Label']

# Convert text to string and handle missing values
X_train = X_train.astype(str).fillna('')
X_val = X_val.astype(str).fillna('')
X_test = X_test.astype(str).fillna('')

# Preprocess data
def preprocess_data(texts, labels, max_length):
    encodings = tokenizer(texts.tolist(), truncation=True, padding='max_length', max_length=max_length, return_tensors='pt')
    labels = torch.tensor(labels.tolist()).to(device)
    dataset = TensorDataset(encodings.input_ids.to(device), encodings.attention_mask.to(device), labels)
    return dataset

max_length = 256  # Set your desired max sequence length
train_dataset = preprocess_data(X_train, y_train, max_length)
val_dataset = preprocess_data(X_val, y_val, max_length)
test_dataset = preprocess_data(X_test, y_test, max_length)

# Create data loaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

# Move model to device
model = model.to(device)

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# Training loop
num_epochs = 20
early_stopping_patience = 3
best_val_loss = float('inf')
epochs_since_last_improvement = 0

for epoch in range(num_epochs):
    model.train()
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids, attention_mask, labels = batch
        input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

    # Validation loop
    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids, attention_mask, labels = batch
            input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            val_loss += outputs.loss.item()
            predicted_labels = torch.argmax(outputs.logits, dim=1)
            correct += (predicted_labels == labels).sum().item()
            total += labels.size(0)

    val_accuracy = correct / total
    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Validation Accuracy BERT Base: {val_accuracy:.4f}")
    print(f"Avg. Validation Loss BERT Base: {avg_val_loss:.4f}")

    # Early stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_since_last_improvement = 0
    else:
        epochs_since_last_improvement += 1
        if epochs_since_last_improvement >= early_stopping_patience:
            print("Early stopping triggered. Stopping training for BERT Base.")
            break


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch [1/20]
Validation Accuracy BERT Base: 0.7117
Avg. Validation Loss BERT Base: 0.7080
Epoch [2/20]
Validation Accuracy BERT Base: 0.8115
Avg. Validation Loss BERT Base: 0.4739
Epoch [3/20]
Validation Accuracy BERT Base: 0.8625
Avg. Validation Loss BERT Base: 0.3652
Epoch [4/20]
Validation Accuracy BERT Base: 0.8897
Avg. Validation Loss BERT Base: 0.3408
Epoch [5/20]
Validation Accuracy BERT Base: 0.9017
Avg. Validation Loss BERT Base: 0.3418
Epoch [6/20]
Validation Accuracy BERT Base: 0.9088
Avg. Validation Loss BERT Base: 0.3086


In [ ]:
# Save the model to Google Drive
from google.colab import drive
drive.mount('/content/drive')

model_path = "/content/drive/MyDrive/Bert.pth"
torch.save(model.state_dict(), model_path)

Mounted at /content/drive


NameError: name 'model' is not defined